In [1]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import AdaBoostClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier


In [3]:
df = pd.read_csv("/content/sample_data/DT_Placement.csv")

print("Dataset:")
print(df)

print("\nDataset Information:")
print(df.info())

Dataset:
     ID  CGPA  Internships Backlogs  Aptitude Placed
0    S1   8.6            2       No        73    Yes
1    S2   9.4            2       No        76    Yes
2    S3   9.0            2       No        81    Yes
3    S4   5.2            0      Yes        48     No
4    S5   5.3            0      Yes        44     No
5    S6   5.0            1       No        83     No
6    S7   6.2            1      Yes        63     No
7    S8   6.2            0      Yes        52    Yes
8    S9   8.7            2      Yes        77    Yes
9   S10   7.5            1       No        70    Yes
10  S11   6.7            1       No        96    Yes
11  S12   7.5            1       No        84    Yes
12  S13   6.8            0      Yes        48     No
13  S14   6.8            0       No        73     No

Dataset Information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14 entries, 0 to 13
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       ------------

In [4]:
df["Backlogs"] = df["Backlogs"].map({
    "No": 0,
    "Yes": 1
})

# Placed: No = 0, Yes = 1
df["Placed"] = df["Placed"].map({
    "No": 0,
    "Yes": 1
})

In [5]:
X = df.drop(["ID", "Placed"], axis=1)

y = df["Placed"]


print("\nFeatures:")
print(X)

print("\nTarget:")
print(y)


Features:
    CGPA  Internships  Backlogs  Aptitude
0    8.6            2         0        73
1    9.4            2         0        76
2    9.0            2         0        81
3    5.2            0         1        48
4    5.3            0         1        44
5    5.0            1         0        83
6    6.2            1         1        63
7    6.2            0         1        52
8    8.7            2         1        77
9    7.5            1         0        70
10   6.7            1         0        96
11   7.5            1         0        84
12   6.8            0         1        48
13   6.8            0         0        73

Target:
0     1
1     1
2     1
3     0
4     0
5     0
6     0
7     1
8     1
9     1
10    1
11    1
12    0
13    0
Name: Placed, dtype: int64


In [6]:
X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y
)


print("\nTraining samples:", len(X_train))
print("Validation samples:", len(X_val))


Training samples: 9
Validation samples: 5


In [7]:
def boosting_benchmark(X_train, y_train, X_val, y_val):

    models = {

        "AdaBoost": AdaBoostClassifier(
            n_estimators=100,
            learning_rate=0.1,
            random_state=42
        ),

        "GBC": GradientBoostingClassifier(
            n_estimators=100,
            learning_rate=0.1,
            max_depth=3,
            random_state=42
        ),

        "XGB": XGBClassifier(
            n_estimators=100,
            learning_rate=0.1,
            max_depth=3,
            random_state=42,
            eval_metric="logloss"
        ),

        "LGB": LGBMClassifier(
            n_estimators=100,
            learning_rate=0.1,
            max_depth=3,
            random_state=42,
            verbosity=-1
        )
    }

    results = []

    for name, model in models.items():

        # Train model
        model.fit(X_train, y_train)

        # Predictions
        train_pred = model.predict(X_train)
        val_pred = model.predict(X_val)

        # Accuracy
        train_accuracy = accuracy_score(
            y_train,
            train_pred
        )

        val_accuracy = accuracy_score(
            y_val,
            val_pred
        )

        results.append({
            "model": name,
            "train_accuracy": train_accuracy,
            "val_accuracy": val_accuracy
        })

    # Convert to DataFrame
    results_df = pd.DataFrame(results)

    # Sort by validation accuracy
    results_df = results_df.sort_values(
        by="val_accuracy",
        ascending=False
    ).reset_index(drop=True)

    return results_df

In [8]:
results = boosting_benchmark(
    X_train,
    y_train,
    X_val,
    y_val
)

In [9]:
print("\nBoosting Benchmark Results:")
print(results)


Boosting Benchmark Results:
      model  train_accuracy  val_accuracy
0  AdaBoost        1.000000           0.8
1       GBC        1.000000           0.6
2       XGB        0.555556           0.6
3       LGB        0.555556           0.6
